# **BioASQ Dataset (Medical QA Corpus)**

**Load Data**

In [34]:
import pandas as pd
import json

# df = pd.read_json(r"..\Datasets\BioASQ\training14b.json")
# df

with open(r"..\Datasets\BioASQ\training14b.json", "r", encoding="utf-8") as f:
    data = json.load(f)
data

{'questions': [{'body': 'Is Hirschsprung disease a mendelian or a multifactorial disorder?',
   'documents': ['http://www.ncbi.nlm.nih.gov/pubmed/15858239',
    'http://www.ncbi.nlm.nih.gov/pubmed/15829955',
    'http://www.ncbi.nlm.nih.gov/pubmed/20598273',
    'http://www.ncbi.nlm.nih.gov/pubmed/6650562',
    'http://www.ncbi.nlm.nih.gov/pubmed/12239580',
    'http://www.ncbi.nlm.nih.gov/pubmed/21995290',
    'http://www.ncbi.nlm.nih.gov/pubmed/15617541',
    'http://www.ncbi.nlm.nih.gov/pubmed/23001136',
    'http://www.ncbi.nlm.nih.gov/pubmed/8896569'],
   'ideal_answer': ["Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model."],
   '

## **Explore Dataset's structure**

**unique keys**

In [35]:
def get_all_keys(d, parent_key=""):
    keys = []
    
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent_key}.{k}" if parent_key else k
            keys.append(full_key)
            keys.extend(get_all_keys(v, full_key))
            
    elif isinstance(d, list):
        for i, item in enumerate(d):
            keys.extend(get_all_keys(item, parent_key))
    
    return list(set(keys))

sorted(get_all_keys(data))

['questions',
 'questions._body',
 'questions._type',
 'questions.body',
 'questions.concepts',
 'questions.documents',
 'questions.duplicate_tmp',
 'questions.duplicate_tmp.from',
 'questions.duplicate_tmp.id',
 'questions.duplicate_tmp.type',
 'questions.exact_answer',
 'questions.id',
 'questions.ideal_answer',
 'questions.snippets',
 'questions.snippets.beginSection',
 'questions.snippets.document',
 'questions.snippets.endSection',
 'questions.snippets.offsetInBeginSection',
 'questions.snippets.offsetInEndSection',
 'questions.snippets.text',
 'questions.triples',
 'questions.triples.o',
 'questions.triples.p',
 'questions.triples.s',
 'questions.type']

In [36]:
print("Number of medical QA pairs :", len(data["questions"]))

Number of medical QA pairs : 5729


**Explore mandatory and optional keys**

In [37]:
from collections import Counter

Counter(len(q.keys()) for q in data["questions"])

Counter({7: 3261, 8: 1249, 6: 946, 9: 273})

**mandatory keys (6 keys)**

In [38]:
for q in data["questions"]:
    if len(q.keys()) == 6:
        essential_keys = q.keys()
        break
essential_keys

dict_keys(['body', 'documents', 'ideal_answer', 'type', 'id', 'snippets'])

## **Create CSV Dataset**

In [39]:
df = pd.DataFrame()
json_df = pd.read_json(r"..\Datasets\BioASQ\training14b.json")
json_df

,questions
0,{'body': 'Is Hirschsprung disease a mendelian ...
1,{'body': 'List signaling molecules (ligands) t...
2,"{'body': 'Is the protein Papilin secreted?', '..."
3,"{'body': 'Are long non coding RNAs spliced?', ..."
4,"{'body': 'Is RANKL secreted from the cells?', ..."
...,...
5724,{'body': 'Is Clostridioides difficile an aerob...
5725,{'body': 'Please list the benefits of sodium-g...
5726,{'body': 'Bacteria that most commonly causes u...
5727,"{'body': 'Is Bacillus Clausii a probiotic?', '..."


In [40]:
import ast

json_df = pd.json_normalize(json_df.iloc[:, 0], errors='ignore')
json_df.tail(5)

,body,documents,ideal_answer,concepts,type,id,snippets,triples,exact_answer,_body,_type,duplicate_tmp.type,duplicate_tmp.from,duplicate_tmp.id
5724,Is Clostridioides difficile an aerobe or a ana...,"[http://www.ncbi.nlm.nih.gov/pubmed/35131507, ...",[Clostridioides difficile is a strict (obligat...,NaN,factoid,67d71d8f18b1e36f2e00002b,"[{'offsetInBeginSection': 121, 'offsetInEndSec...",NaN,[strict anaerobe],NaN,NaN,NaN,NaN,NaN
5725,Please list the benefits of sodium-glucose cot...,"[http://www.ncbi.nlm.nih.gov/pubmed/29485012, ...","[helps manage type 2 diabetes, promotes weight...",NaN,list,67cdb26a81b1027333000017,"[{'offsetInBeginSection': 1576, 'offsetInEndSe...",NaN,"[[reduce hospitalization], [weight loss], [imp...",NaN,NaN,NaN,NaN,NaN
5726,Bacteria that most commonly causes urinary tra...,"[http://www.ncbi.nlm.nih.gov/pubmed/18703368, ...","[E. coli, Klebsiella spp, Staphylococcus sapro...",NaN,factoid,67e2b0a018b1e36f2e00008b,"[{'offsetInBeginSection': 1010, 'offsetInEndSe...",NaN,[Escherichia coli],NaN,NaN,NaN,NaN,NaN
5727,Is Bacillus Clausii a probiotic?,"[http://www.ncbi.nlm.nih.gov/pubmed/23816335, ...","[Yes, Bacillus Clausii is a spore-forming bact...",NaN,yesno,67e090be18b1e36f2e00006c,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN,yes,NaN,NaN,NaN,NaN,NaN
5728,Symptoms of chlamydia genital infection in women.,"[http://www.ncbi.nlm.nih.gov/pubmed/36759099, ...",[Symptoms of Chlamydia genital infection in wo...,NaN,list,67d7fded18b1e36f2e000042,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN,"[[asymptomatic], [lower abdominal discomfort],...",NaN,NaN,NaN,NaN,NaN


In [41]:
json_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5729 entries, 0 to 5728
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   body                5729 non-null   str    
 1   documents           5729 non-null   object 
 2   ideal_answer        5729 non-null   object 
 3   concepts            1854 non-null   object 
 4   type                5729 non-null   str    
 5   id                  5729 non-null   str    
 6   snippets            5729 non-null   object 
 7   triples             322 non-null    object 
 8   exact_answer        4366 non-null   object 
 9   _body               18 non-null     str    
 10  _type               17 non-null     str    
 11  duplicate_tmp.type  1 non-null      float64
 12  duplicate_tmp.from  1 non-null      str    
 13  duplicate_tmp.id    1 non-null      object 
dtypes: float64(1), object(7), str(6)
memory usage: 1.1+ MB


In [42]:
type(json_df["ideal_answer"].iloc[0])

list

In [43]:
df["question"] = json_df["body"]
df["answer"] = json_df["ideal_answer"].apply(lambda x: x[0])
df["type"] = json_df["type"]

def process_answer(row):
    if row["type"] == "list":
        return " | ".join(i[0] for i in row["exact_answer"])
    elif row["type"] == "factoid":
        return row["exact_answer"][0]
    elif row["type"] == "yesno":
        return row["exact_answer"]
    else:
        return None

df["exact_answer"] = json_df.apply(process_answer, axis=1)

def process_snippets(snippets):
    extracted_context = ""
    for snippet in snippets:
        snip_text = snippet["text"]
        snip_text = snip_text.strip()
        extracted_context += snip_text + "\n"
    return extracted_context

df["context"] = json_df["snippets"].apply(process_snippets)
df = df[["context", "question", "answer", "exact_answer", "type"]]
df

,context,question,answer,exact_answer,type
0,Hirschsprung disease (HSCR) is a multifactoria...,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...",NaN,summary
1,the epidermal growth factor receptor (EGFR) li...,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,epidermal growth factor | betacellulin | epire...,list
2,"Using expression analysis, we identify three g...",Is the protein Papilin secreted?,"Yes, papilin is a secreted protein",yes,yesno
3,Our analyses indicate that lncRNAs are generat...,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,yes,yesno
4,Osteoprotegerin (OPG) is a soluble secreted fa...,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,yes,yesno
...,...,...,...,...,...
5724,strictly anaerobic bacterium\nClostridioides d...,Is Clostridioides difficile an aerobe or a ana...,Clostridioides difficile is a strict (obligate...,strict anaerobe,factoid
5725,Glucagon antagonism enhances the therapeutic e...,Please list the benefits of sodium-glucose cot...,"helps manage type 2 diabetes, promotes weight ...",reduce hospitalization | weight loss | improve...,list
5726,Escherichia coli was the most identified uropa...,Bacteria that most commonly causes urinary tra...,"E. coli, Klebsiella spp, Staphylococcus saprop...",Escherichia coli,factoid
5727,Bacillus clausii is a commercial spore probiot...,Is Bacillus Clausii a probiotic?,"Yes, Bacillus Clausii is a spore-forming bacte...",yes,yesno


In [44]:
print(df["context"].iloc[0])

Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes
In this study, we review the identification of genes and loci involved in the non-syndromic common form and syndromic Mendelian forms of Hirschsprung's disease. The majority of the identified genes are related to Mendelian syndromic forms of Hirschsprung's disease. The non-Mendelian inheritance of sporadic non-syndromic Hirschsprung's disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model
Coding sequence mutations in e.g. RET, GDNF, EDNRB, EDN3, and SOX10 lead to long-segment (L-HSCR) as well as syndromic HSCR but fail to explain the transmission of the much more common short-segment form (S-HSCR). Furthermore, mutations in the RET gene are responsible for approximately half of the familial and some sporadic cases, 

In [45]:
df.to_csv(r"..\Datasets\BioASQ\bioasq_dataset.csv", index=False)